# 🔗 Session Affinity - Validation Tests

## Overview

Validate **backend-pool session affinity (sticky routing)** for stateful models through the Unified AI API's OpenAI **Responses API** (`/unified-ai/openai/responses`).

When a model is served by a **pool** (2+ backends) and marked `sessionAwareModel: true`, APIM emits a `Set-Cookie: ai-gateway-affinity=...` header on the first response and routes any request that **replays that cookie** back to the **same backend**. This is required for the stateful Responses API, where a follow-up call (referencing a `previous_response_id`) must reach the backend that holds the conversation state.

This notebook:
- **Enables session affinity** for the target model by regenerating the LLM backend onboarding (`sessionAwareModel: true`) into a `-local.bicepparam` and redeploying it (only when backends come from `azd`).
- **Provisions an access contract** (APIM product + subscription) scoped to the target model, with `enableResponseHeaders = true` so the responding backend is observable.
- **Establishes 4 sessions × 3 requests** using the **OpenAI SDK** over an **`httpx` client with its own cookie jar** per session — the first request creates a response id, and the 2 follow-ups reuse it (via `previous_response_id`) while the cookie jar keeps them pinned to the same backend.
- **Visualizes** the response id, responding backend, and session cookie for every request.
- **Asserts** that within each session the response id is reused, the same backend answers all 3 requests, and the affinity cookie is maintained.

> **Target model** defaults to `gpt-5.2` (pooled across 2 backends and supports the stateful Responses API). Set `target_model` to any onboarded model that has a pool and session affinity enabled.

> **Prerequisites:** A Citadel Governance Hub with the Unified AI API enabled; the target model onboarded across **2+ backends**; `openai`, `httpx`, and `pandas` installed (`pip install -r ../shared/requirements.txt`). See [LLM Backend Onboarding - Session Affinity Configuration](../bicep/infra/llm-backend-onboarding/README.md#session-affinity-configuration) and [LLM Access Guide - Session affinity](../guides/llm-access-guide.md#backend-pool-types).


### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, and `llm_backends_config` from your active `azd` environment.
- `False` — fill the `REPLACE` values manually below.

Set `target_model` to a pooled, session-aware model. Adjust `num_sessions` / `requests_per_session` to change the test matrix.


In [ ]:
import os
import sys, json, time, uuid
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION (REQUIRED — used as defaults if azd lookup fails)
# ============================================================================
governance_hub_resource_group = "REPLACE"   # Resource group of the deployed Citadel Governance Hub
location = "REPLACE"                         # Azure region (e.g. "swedencentral", "eastus")
llm_backends_config = []                     # Existing backends (auto-loaded from azd LLM_BACKEND_CONFIG)

# ============================================================================
# 🔗 SESSION AFFINITY TEST CONFIGURATION
# ============================================================================
target_model = "gpt-5.2"        # Pooled (2+ backends) + stateful Responses API model to test
num_sessions = 4                # How many independent sticky sessions to establish
requests_per_session = 3        # Requests per session (1 create + follow-ups reusing the response id)
session_cookie_name = "ai-gateway-affinity"   # Affinity cookie name (matches sessionAffinityDefaults)

# ============================================================================
# 🛠️ BACKEND ONBOARDING (enable session affinity for target_model)
# ----------------------------------------------------------------------------
# When backends are loaded from azd, this notebook regenerates the LLM backend
# onboarding with `sessionAwareModel: true` on target_model and redeploys it.
# Set False if the target model is already onboarded with session affinity.
# ============================================================================
run_backend_onboarding = True

# ============================================================================
# 📜 ACCESS CONTRACT CONFIGURATION (created on the fly by this notebook)
# ============================================================================
contract_business_unit = "Testing"
contract_use_case_name = "SessionAffinity"
contract_environment   = "DEV"

# ============================================================================
# 🧹 CLEANUP (disabled by default) — deletes ONLY the access contract product,
# never the backend onboarding. Flip to True in the cleanup cell to remove it.
# ============================================================================
cleanup_delete_contract = False

# Conversation prompts. Request 1 creates the response; requests 2+ reuse the
# first response id (previous_response_id) so they must land on the same backend.
session_prompts = [
    "My name is Ada and my favorite number is 7. Reply with a short acknowledgement.",
    "What name did I tell you? Answer in one short sentence.",
    "What is my favorite number multiplied by 2? Answer with just the number.",
]

# ============================================================================
# 🔁 azd ENVIRONMENT OVERRIDES
# ============================================================================
if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group": ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":       ["AZURE_LOCATION", "LOCATION"],
        "llm_backends":   (["LLM_BACKEND_CONFIG", "LLM_BACKENDS_CONFIG"], "json"),
    }, verbose=False)
    if "resource_group" in loaded:  governance_hub_resource_group = loaded["resource_group"]
    if "location" in loaded:        location = loaded["location"]
    if "llm_backends" in loaded:    llm_backends_config = loaded["llm_backends"]
    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")
    utils.print_ok(f"Backends loaded: {len(llm_backends_config)}")

utils.print_ok("Notebook variables initialized!")


### 1️⃣ Verify Azure CLI and Connected Subscription


In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")


### 2️⃣ Initialize APIM Client Tool

Discover the Unified AI API, capture the gateway URL, and detect the managed identity used by backend onboarding.


In [ ]:
try:
    apimClientTool = APIMClientTool(governance_hub_resource_group)
    apimClientTool.initialize()
    apimClientTool.discover_api("unified-ai")

    gateway_url = str(apimClientTool.apim_resource_gateway_url).rstrip("/")
    utils.print_info(f"Gateway URL: {gateway_url}")

    # Managed identity used by the onboarding contract
    mi = apimClientTool.get_managed_identity_info()
    managed_identity_name           = mi.get('name')
    managed_identity_resource_group = mi.get('resourceGroup') or governance_hub_resource_group
    utils.print_info(f"Managed Identity: {managed_identity_name} (rg={managed_identity_resource_group})")

    # Models that currently have a pool (candidates for session affinity)
    try:
        pooled_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
        utils.print_info(f"Models in set-backend-pools fragment: {pooled_models}")
    except Exception as e:
        utils.print_warning(f"Could not read set-backend-pools fragment yet: {e}")

    utils.print_ok("APIM Client Tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")


### 2️⃣B Enable Session Affinity for the Target Model (Backend Onboarding)

Session affinity only activates when the model is served by a **pool** (2+ backends) **and** marked `sessionAwareModel: true`. This step takes the `llm_backends_config` loaded from `azd`, sets `sessionAwareModel: true` on **every backend that serves `target_model`**, and redeploys the LLM Backend Onboarding Bicep.

The generated parameter file is written with a **`-local.bicepparam`** suffix (git-ignored) and includes `configureSessionAffinity = true` + `sessionAffinityDefaults` (cookie `ai-gateway-affinity`). All other backends/models are preserved unchanged. Skipped when `run_backend_onboarding = False` or when no azd backends are available.


In [ ]:
import copy

def _norm_models(models):
    return [dict(m) if isinstance(m, dict) else {"name": m} for m in models]

onboard_cfg = copy.deepcopy(llm_backends_config) if llm_backends_config else []

# Mark target_model session-aware on every backend that serves it.
backends_serving = 0
for b in onboard_cfg:
    models = _norm_models(b.get('supportedModels', []))
    touched = False
    for m in models:
        if m.get('name') == target_model:
            m['sessionAwareModel'] = True
            touched = True
    if touched:
        b['supportedModels'] = models
        backends_serving += 1

if onboard_cfg:
    if backends_serving == 0:
        utils.print_warning(f"'{target_model}' was not found on any backend in llm_backends_config — "
                            f"add it (or pick another model) before testing session affinity.")
    elif backends_serving == 1:
        utils.print_warning(f"'{target_model}' is served by only ONE backend — session affinity needs a POOL "
                            f"(2+ backends). The test will run but routing cannot vary between backends.")
    else:
        utils.print_ok(f"Marked '{target_model}' sessionAwareModel=true on {backends_serving} backends "
                       f"(pool → session affinity will be emitted).")


def _fmt_model(m):
    parts = [f"name: '{m['name']}'"]
    for k in ('sku', 'modelFormat', 'modelVersion', 'retirementDate', 'apiVersion', 'inferenceApiVersion', 'modelPath'):
        if m.get(k) not in (None, ''):
            parts.append(f"{k}: '{m[k]}'")
    for k in ('capacity', 'timeout'):
        if m.get(k) not in (None, ''):
            parts.append(f"{k}: {m[k]}")
    if m.get('sessionAwareModel'):
        parts.append("sessionAwareModel: true")
    return "{ " + ", ".join(parts) + " }"


def _fmt_backend(b):
    models = "\n      ".join(_fmt_model(m if isinstance(m, dict) else {"name": m}) for m in b['supportedModels'])
    auth = f"    authType: '{b['authType']}'\n" if b.get('authType') else ""
    return (f"  {{\n    backendId: '{b['backendId']}'\n    backendType: '{b['backendType']}'\n"
            f"    endpoint: '{b['endpoint']}'\n{auth}    supportedModels: [\n      {models}\n    ]\n"
            f"    priority: {b.get('priority', 1)}\n    weight: {b.get('weight', 100)}\n  }}")


if not run_backend_onboarding:
    utils.print_warning("run_backend_onboarding = False — skipping onboarding (assuming target model is already session-aware).")
elif not onboard_cfg:
    utils.print_warning("llm_backends_config is empty (no azd LLM_BACKEND_CONFIG) — cannot regenerate onboarding. "
                        "Set init_from_azd=True with a deployed backend config, or ensure the model is already session-aware.")
else:
    onboarding_dir = "../bicep/infra/llm-backend-onboarding"
    onboarding_params = os.path.join(onboarding_dir, "llm-backends-session-affinity-validation-local.bicepparam")
    backends_str = "\n".join(_fmt_backend(b) for b in onboard_cfg)

    onboarding_content = f"""using './main.bicep'

// Generated by citadel-session-affinity-tests.ipynb @ {time.strftime('%Y-%m-%d %H:%M:%S')}
// Enables session affinity (sticky routing) for '{target_model}' by setting
// sessionAwareModel: true on every backend that serves it. All other backends
// and models are preserved from the azd LLM_BACKEND_CONFIG.

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param apimManagedIdentity = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{managed_identity_resource_group}'
  name: '{managed_identity_name}'
}}

param llmBackendConfig = [
{backends_str}
]

param configureCircuitBreaker = true

// Session affinity — global kill-switch on; the real opt-in is the per-model
// sessionAwareModel flag set above. Cookie defaults to 'ai-gateway-affinity'.
param configureSessionAffinity = true
param sessionAffinityDefaults = {{
  cookieName: '{session_cookie_name}'
  source: 'Cookie'
}}
"""

    with open(onboarding_params, 'w') as fh:
        fh.write(onboarding_content)
    utils.print_ok(f"Wrote {onboarding_params}")
    print("\n" + "-" * 60)
    print(onboarding_content)
    print("-" * 60)

    onboarding_name = f"session-affinity-backends-{time.strftime('%Y%m%d%H%M%S')}"
    onboarding_template = os.path.join(onboarding_dir, "main.bicep")
    onboarding_cmd = (
        f"az deployment sub create --name {onboarding_name} "
        f"--location {location} --template-file {onboarding_template} --parameters {onboarding_params}"
    )
    out = utils.run(onboarding_cmd, f"Onboarding '{onboarding_name}' succeeded", f"Onboarding '{onboarding_name}' failed")
    if out.success:
        utils.print_ok(f"✅ '{target_model}' onboarded with session affinity — pool now emits the '{session_cookie_name}' cookie.")
        apimClientTool.discover_api("unified-ai")


### 3️⃣ Provision Access Contract

Deploy an access contract (APIM product + subscription) scoped to `target_model`. The product policy uses `set-llm-requested-model` and enforces `allowedModels`, and sets `enableResponseHeaders = true` so responses expose the resolved backend via `UAIG-Backend` (and `x-ms-region`).


In [ ]:
bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

timestamp = time.strftime('%Y%m%d%H%M%S')
contract_name = f"session-affinity-test-{timestamp}"

folder_name = f"{contract_business_unit.lower()}-{contract_use_case_name.lower()}"
environment_folder = contract_environment.lower()
contract_folder = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
os.makedirs(contract_folder, exist_ok=True)
utils.print_info(f"📁 Created folder: {contract_folder}")

allowed_models_csv = target_model
utils.print_info(f"Allowed models for access contract: {allowed_models_csv}")

product_policy = f"""<policies>
    <inbound>
        <base />
        <!-- Extract and validate the requested model (body / path / x-ai-model header) -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Restrict access to only the target image/LLM model under test -->
        <set-variable name="allowedModels" value="{allowed_models_csv}" />

        <!-- Emit UAIG-* response headers (UAIG-Backend, etc.) so the backend is observable -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>"""

policy_file_dest = os.path.join(contract_folder, "ai-product-policy.xml")
with open(policy_file_dest, 'w') as f:
    f.write(product_policy)
utils.print_ok(f"📋 Custom policy file created: {policy_file_dest}")

params_file = os.path.join(contract_folder, "main.bicepparam")
params_content = f"""using '../../../main.bicep'

// Session Affinity Test Contract - Generated from Notebook
param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  name: 'placeholder'
}}

param useTargetAzureKeyVault = false

param useCase = {{
  businessUnit: '{contract_business_unit}'
  useCaseName: '{contract_use_case_name}'
  environment: '{contract_environment}'
}}

param apiNameMapping = {{
  LLM: ['unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: 'SESSION-AFFINITY-TEST-ENDPOINT'
    apiKeySecretName: 'SESSION-AFFINITY-TEST-KEY'
    policyXml: loadTextContent('ai-product-policy.xml')
  }}
]

param productTerms = 'Session affinity test contract - generated from validation notebook'

param useTargetFoundry = false

param foundry = {{
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}}
"""

with open(params_file, 'w') as f:
    f.write(params_content)
utils.print_ok(f"✅ Parameter file created: {params_file}")

product_id = f"LLM-{contract_business_unit}-{contract_use_case_name}-{contract_environment}"
utils.print_info(f"Deploying access contract: {contract_name} (Product: {product_id})...")

deployment_cmd = f"az deployment sub create --name {contract_name} --location {location} --template-file {template_file} --parameters {params_file}"
output = utils.run(deployment_cmd, f"Deployment '{contract_name}' succeeded", f"Deployment '{contract_name}' failed")

if output.success:
    utils.print_ok(f"✅ Access contract deployed! Product ID: {product_id}")
else:
    utils.print_error("❌ Access contract deployment failed!")

apimClientTool.initialize()


### 4️⃣ Retrieve API Key & Build the OpenAI Base URL

Pull the contract's subscription key and compute the base URL for the OpenAI SDK. The SDK appends `/responses`, so the base URL points at the Unified AI OpenAI-compatible surface: `POST {gateway}/unified-ai/openai/responses`.


In [ ]:
subscription_name = f"{product_id}-SUB-01"

api_key = None
for sub in apimClientTool.apim_subscriptions:
    if subscription_name.lower() in sub.get('name', '').lower():
        api_key = sub.get('key')
        utils.print_ok(f"Found API key from subscription: {sub.get('name')}")
        break

if not api_key:
    raise Exception(f"No API key found for subscription '{subscription_name}'. Ensure the access contract deployed successfully.")

# OpenAI Responses API base URL via the Unified AI API (OpenAI-compatible surface).
# The SDK appends '/responses' → POST {gateway}/unified-ai/openai/responses
openai_base_url = f"{gateway_url}/unified-ai/openai"
utils.print_info(f"OpenAI base_url: {openai_base_url}")
utils.print_ok("API key and endpoint configured successfully!")


### 5️⃣ Stateful Session Client & Test Harness

Each **session** gets its own `httpx.Client` (a private cookie jar) wrapped by an `OpenAI` client. Because `httpx` stores `Set-Cookie` and replays it automatically, every request in a session carries the `ai-gateway-affinity` cookie set by the first response — pinning the whole session to one backend.

For each request we use `client.responses.with_raw_response.create(...)` so we can read both the parsed response (its `id`) **and** the raw headers (`UAIG-Backend`, `x-ms-region`, `Set-Cookie`). The first request creates the anchor response id; the follow-ups reuse it via `previous_response_id`, so they only succeed if they reach the backend that stored it.


In [ ]:
import httpx
from openai import OpenAI


def _backend_fingerprint(headers):
    """Identify which backend answered. UAIG-Backend is the resolved pool id; x-ms-region
    distinguishes pool members that live in different regions (the common HA layout)."""
    pool   = headers.get('UAIG-Backend', 'n/a')
    region = headers.get('x-ms-region', 'n/a')
    return pool, region


def make_session_client():
    """A fresh OpenAI client backed by its own httpx cookie jar = one sticky session."""
    http_client = httpx.Client(timeout=120.0)   # persists Set-Cookie across calls in this session
    client = OpenAI(
        api_key="apim-uses-subscription-key",   # placeholder; APIM authenticates via the api-key header
        base_url=openai_base_url,
        http_client=http_client,
        default_headers={"api-key": api_key},
        max_retries=0,                            # no SDK retries — keep routing observations clean
    )
    return client, http_client


def run_session(session_idx):
    """Run one sticky session of `requests_per_session` Responses API calls and return per-request records."""
    client, http_client = make_session_client()
    records = []
    anchor_id = None
    for req_idx in range(requests_per_session):
        prompt = session_prompts[req_idx % len(session_prompts)]
        kwargs = {"model": target_model, "input": prompt, "store": True}
        if req_idx > 0 and anchor_id:
            kwargs["previous_response_id"] = anchor_id   # follow-ups reuse the FIRST response id
        prev_used = kwargs.get("previous_response_id", "")

        status, resp_id, pool, region, err = 0, "", "n/a", "n/a", ""
        try:
            raw = client.responses.with_raw_response.create(**kwargs)
            status = getattr(raw, "status_code", None) or raw.http_response.status_code
            pool, region = _backend_fingerprint(raw.headers)
            parsed = raw.parse()
            resp_id = getattr(parsed, "id", "") or ""
            if req_idx == 0:
                anchor_id = resp_id
        except Exception as e:
            err = str(e)[:300]
            resp = getattr(e, "response", None)
            if resp is not None:
                status = getattr(resp, "status_code", 0)
                try:
                    pool, region = _backend_fingerprint(resp.headers)
                except Exception:
                    pass
            utils.print_error(f"[session {session_idx + 1} req {req_idx + 1}] {err}")

        cookie = http_client.cookies.get(session_cookie_name) or ""
        records.append({
            "session": session_idx + 1,
            "request": req_idx + 1,
            "status": status,
            "response_id": resp_id,
            "previous_response_id": prev_used,
            "anchor_response_id": anchor_id or "",
            "backend_pool": pool,
            "backend_region": region,
            "cookie": cookie,
            "error": err,
        })
        short = (resp_id[:22] + "…") if len(resp_id) > 23 else resp_id
        utils.print_info(f"  session {session_idx + 1} · req {req_idx + 1} → {status} | "
                         f"id={short or 'n/a'} | region={region} | cookie={'set' if cookie else 'none'}")
    http_client.close()
    return records


### 6️⃣ Run the Sessions (4 sessions × 3 requests)

Each session is independent (its own cookie jar). Watch the `region` and `cookie` stay constant within a session.


In [ ]:
all_records = []
for s in range(num_sessions):
    utils.print_info(f"▶ Session {s + 1}/{num_sessions}")
    all_records.extend(run_session(s))

utils.print_ok(f"Completed {num_sessions} sessions × {requests_per_session} requests = {len(all_records)} requests.")


### 7️⃣ Visualize — Response ID, Backend & Cookie per Request

Every row is one request. `Region`/`Backend Pool` identify the responding backend; `Affinity Cookie` is the value stored in the session's cookie jar.


In [ ]:
def _short(v, n=18):
    v = str(v or "")
    return v if len(v) <= n else v[:n] + "…"

rows = [{
    "Session": r["session"],
    "Req": r["request"],
    "Status": r["status"],
    "Response ID": _short(r["response_id"], 26),
    "Prev Resp ID": _short(r["previous_response_id"], 26),
    "Backend Pool": _short(r["backend_pool"], 24),
    "Region": r["backend_region"],
    "Affinity Cookie": _short(r["cookie"], 20),
} for r in all_records]

try:
    import pandas as pd
    from IPython.display import display
    df = pd.DataFrame(rows)
    display(df.style.hide(axis="index").set_caption(
        f"Session affinity — {target_model} ({num_sessions} sessions × {requests_per_session} requests)"))
except Exception:
    hdr = list(rows[0].keys()) if rows else []
    print(" | ".join(hdr))
    print("-" * 100)
    for row in rows:
        print(" | ".join(str(row[h]) for h in hdr))


### 8️⃣ Assertions — Affinity Held

For every session we assert:
1. **All requests succeeded** (`status == 200`) — follow-ups referencing the first response id only succeed on the backend that stored it.
2. **Response id reused** — the anchor id is non-empty and both follow-ups sent it as `previous_response_id`.
3. **Same backend** — one distinct `x-ms-region` (backend) across the session.
4. **Affinity cookie maintained** — exactly one `ai-gateway-affinity` cookie value across the session.

> **Note:** If all pool members share the same Azure region, `x-ms-region` can't distinguish them — in that case the stable affinity cookie + the successful `previous_response_id` reuse are the affinity evidence.


In [ ]:
from collections import defaultdict

by_session = defaultdict(list)
for r in all_records:
    by_session[r["session"]].append(r)

failures = []
for sid, recs in sorted(by_session.items()):
    recs = sorted(recs, key=lambda x: x["request"])
    first = recs[0]
    anchor = first["response_id"]
    non200 = [r["request"] for r in recs if r["status"] != 200]
    followups_ref_anchor = all(r["previous_response_id"] == anchor for r in recs[1:]) if len(recs) > 1 else True
    regions = {r["backend_region"] for r in recs}
    cookies = {r["cookie"] for r in recs if r["cookie"]}

    ok = (not non200) and bool(anchor) and followups_ref_anchor and len(regions) == 1 and len(cookies) == 1
    (utils.print_ok if ok else utils.print_error)(
        f"[Session {sid}] {'PASS' if ok else 'FAIL'} | anchor={(anchor[:20] + '…') if anchor else 'n/a'} | "
        f"followups→anchor={followups_ref_anchor} | regions={sorted(regions)} | "
        f"cookies={len(cookies)} | non200={non200}")
    if not cookies:
        utils.print_warning(f"[Session {sid}] no '{session_cookie_name}' cookie was set — is the model pooled AND sessionAwareModel=true?")
    if not ok:
        failures.append(sid)

print()
if failures:
    utils.print_error(f"❌ Session affinity assertions FAILED for sessions: {failures}")
else:
    utils.print_ok(f"✅ All {len(by_session)} sessions passed — response id reused, same backend, affinity cookie maintained.")


### 9️⃣ Summary Table

One row per session: the anchor **response id**, the **backend** that served it, and the **session cookie**.


In [ ]:
summary = []
for sid, recs in sorted(by_session.items()):
    recs = sorted(recs, key=lambda x: x["request"])
    first = recs[0]
    regions = sorted({r["backend_region"] for r in recs})
    cookies = sorted({r["cookie"] for r in recs if r["cookie"]})
    cookie_disp = "none"
    if cookies:
        cookie_disp = cookies[0] if len(cookies[0]) <= 30 else cookies[0][:29] + "…"
    summary.append({
        "Session": sid,
        "Response ID (anchor)": first["response_id"] or "n/a",
        "Backend Region": regions[0] if len(regions) == 1 else ",".join(regions),
        "Affinity Cookie": cookie_disp,
        "Requests": len(recs),
        "All 200": all(r["status"] == 200 for r in recs),
    })

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(summary).style.hide(axis="index").set_caption("Session Affinity Summary"))
except Exception:
    for s in summary:
        print(s)


### 🔟 Cleanup (disabled by default)

Deletes **only** the access contract product and its subscription. The **backend onboarding is left intact** (the target model stays session-aware). Set `cleanup_delete_contract = True` in the init cell to enable.


In [ ]:
if not cleanup_delete_contract:
    utils.print_warning("cleanup_delete_contract = False — skipping cleanup.")
    utils.print_info(f"Would delete APIM product '{product_id}' (and its subscription). "
                     f"The session-affinity backend onboarding is always left intact.")
else:
    utils.print_info(f"Deleting access contract product '{product_id}' (and subscriptions)...")
    try:
        apimClientTool.client.product.delete(
            resource_group_name=apimClientTool.resource_group_name,
            service_name=apimClientTool.apim_resource_name,
            product_id=product_id,
            if_match="*",
            delete_subscriptions=True,
        )
        utils.print_ok(f"✅ Deleted product '{product_id}' and its subscription. "
                       f"Backend onboarding (session affinity) left untouched.")
    except Exception as e:
        utils.print_error(f"Cleanup failed: {e}")


## ✅ Summary

This notebook validated **backend-pool session affinity** for the stateful OpenAI Responses API:
- Enabled `sessionAwareModel: true` for the target model and redeployed the LLM backend onboarding (`-local.bicepparam`).
- Provisioned an access contract scoped to the target model with response headers enabled.
- Ran 4 sessions × 3 requests using the OpenAI SDK over a per-session `httpx` cookie jar; the first request created a response id and the follow-ups reused it.
- Visualized the response id, backend, and affinity cookie per request, and asserted that each session stuck to one backend with a maintained `ai-gateway-affinity` cookie.

See [LLM Backend Onboarding — Session Affinity Configuration](../bicep/infra/llm-backend-onboarding/README.md#session-affinity-configuration) and [LLM Access Guide — Backend Pool Types](../guides/llm-access-guide.md#backend-pool-types) for the underlying configuration and client cookie-jar guidance.
